# Real-Time Mode in Structured Streaming

All the queries in this chapter so far ran as **micro-batches**: collect the
rows that arrived, process them as a small batch job, commit, repeat. That
is efficient and easy to recover, but every batch pays a fixed overhead, so
latency cannot drop below a certain floor. **Real-time mode** is a
Structured Streaming trigger type that removes that floor, with end-to-end
latency as low as five milliseconds
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#what-is-real-time-mode)),
while keeping the same DataFrame API. Apache Spark first supported it in
version 4.1.0
([Apache Spark: Spark Release 4.1.0](https://spark.apache.org/releases/spark-release-4.1.0.html)),
and on Databricks it runs on classic compute
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#prerequisites)).

This notebook explains how real-time mode works, how to write and size a
real-time query, and when it is worth its cost. Real-time mode does not run
on serverless compute, which this course uses, so most code on this page is
shown rather than executed.

## The Latency Floor of Micro-Batches

{doc}`Introduction to Stream Processing <01_introduction_to_stream_processing>`
compared the two processing models. Micro-batching achieves high throughput
and simple recovery because each batch is a self-contained unit of work. But
each batch also has to be planned, scheduled as tasks, written to the sink
and committed to the checkpoint before the next one starts. The progress
report in
{doc}`Spark Structured Streaming <02_spark_structured_streaming>` broke a
batch duration down into exactly these phases: `queryPlanning`,
`latestOffset`, `addBatch`, `walCommit`, `commitBatch` and so on. Even for a
few hundred rows, a micro-batch took seconds there.

A record that arrives just after a batch started has to wait for that batch
to finish, and then for the whole next batch. For a dashboard that refreshes
every minute this does not matter. For fraud detection, where a payment must
be approved or blocked while the customer waits, it does.

## How Real-Time Mode Works

Real-time mode changes three things about how a query executes
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#how-real-time-mode-achieves-low-latency)):

- **Long-running batches.** Instead of short batches that each read a fixed
  set of new records, a batch runs for a fixed duration, five minutes by
  default, and processes data as it arrives in the source during that time.
- **Concurrent stages.** In micro-batch mode, a stage runs after the stage
  before it has finished. In real-time mode, all stages of the query are
  scheduled simultaneously.
- **Streaming shuffle.** Data is passed between stages with a streaming
  shuffle, instead of being written out in full before the next stage reads
  it.

```mermaid
:caption: In micro-batch mode, each record waits for its batch to be planned, executed and committed. In real-time mode, the stages run side by side and a record flows through them as soon as it arrives. Progress is committed only between long-running batches.
:align: center

graph LR
    subgraph Micro-batch
        M1["Plan batch"] --> M2["Run stage 1"] --> M3["Run stage 2"] --> M4["Commit"] --> M1
    end
    subgraph Real-time
        R1(["Record"]) --> R2["Stage 1, always running"] --> R3["Stage 2, always running"] --> R4(["Sink"])
    end
```

The batch length sets the checkpoint frequency. It is therefore not a
latency setting but a trade-off: longer batches mean slower replay after a
failure and metrics that arrive later. Databricks recommends benchmarking
real-time mode against your target workload to choose the interval
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#how-real-time-mode-achieves-low-latency)).

## Requirements on Databricks

Real-time mode places strict requirements on the compute it runs on
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#prerequisites),
[Real-time mode reference](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference#compute-types)):

| Setting | Requirement |
| --- | --- |
| Compute | Classic compute. Serverless compute is not supported. |
| Access mode | Dedicated, or Standard for Python queries only |
| Databricks Runtime | 16.4 LTS or above |
| Autoscaling | Off |
| Photon | Off |
| Spot instances | Off, so that executors are not interrupted |
| Spark configuration | `spark.databricks.streaming.realTimeMode.enabled true` |

The cluster runs all the time, and every stage of the query permanently
occupies task slots. That makes a real-time query considerably more
expensive than a scheduled `availableNow` job processing the same data.

## Writing a Real-Time Query

A real-time query is an ordinary streaming query with two changes: the
trigger is `realTime`
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#query-configuration)),
and the output mode must be `update`, the only mode real-time mode supports
([Real-time mode reference](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)).
The duration passed to the trigger is the length of the long-running batch.

The example below reads bakery sales from a Kafka topic, flags large orders
and publishes them to a second topic, where a service that alerts the
bakery staff is listening. It only uses operations that real-time mode
supports: parsing, filtering, a projection and a broadcast join with a small
static table.

```python
from pyspark.sql import functions as F

SALES_SCHEMA = """
    transactionID BIGINT, customerID BIGINT, franchiseID BIGINT, dateTime TIMESTAMP,
    product STRING, quantity BIGINT, unitPrice BIGINT, totalPrice BIGINT, paymentMethod STRING
"""

franchises = spark.read.table("samples.bakehouse.sales_franchises").select("franchiseID", "name", "city")

large_orders = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092,broker-2:9092")
    .option("subscribe", "bakery-sales")
    .option("maxPartitions", 8)
    .load()
    .select(F.from_json(F.col("value").cast("string"), SALES_SCHEMA).alias("sale"))
    .select("sale.*")
    .filter(F.col("quantity") >= 30)
    .join(F.broadcast(franchises), "franchiseID")
    .select(
        F.col("transactionID").cast("string").alias("key"),
        F.to_json(F.struct("transactionID", "name", "city", "product", "quantity", "dateTime")).alias("value"),
    )
)

query = (
    large_orders.writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092,broker-2:9092")
    .option("topic", "large-orders")
    .option("checkpointLocation", "/Volumes/main/bakery/checkpoints/large_orders")
    .outputMode("update")
    .trigger(realTime="5 minutes")
    .start()
)
```

Apart from the trigger and the output mode, this is the same code you would
write for a micro-batch query. Switching an existing query between the two
modes is mostly a matter of changing those two lines.

### Trying It on Serverless Compute

Starting the same kind of query on this course's serverless compute shows
the compute requirement in action. The query is rejected before it ever
contacts a Kafka broker:

In [1]:
from pyspark.sql import functions as F

from project.utils.sparksession import get_spark_session

spark = get_spark_session(profile="free-edition")

In [2]:
(
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092")
    .option("subscribe", "bakery-sales")
    .load()
    .writeStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker-1:9092")
    .option("topic", "large-orders")
    .option("checkpointLocation", "/Volumes/main/bakery/checkpoints/large_orders")
    .outputMode("update")
    .trigger(realTime="5 minutes")
    .start()
)

AnalysisException: [UC_COMMAND_NOT_SUPPORTED_IN_SHARED_ACCESS_MODE] The command(s): Real Time Mode (RealTimeTrigger) are not supported for Unity Catalog clusters in shared access mode. Use single user access mode instead. SQLSTATE: 0AKUC

JVM stacktrace:
org.apache.spark.sql.AnalysisException
	at com.databricks.sql.managedcatalog.ManagedCatalogErrors$.operationNotSupportedForSharedAccessModeException(ManagedCatalogErrors.scala:247)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.handleWriteStreamOperationStart(SparkConnectPlanner.scala:5010)
	at org.apache.spark.sql.connect.planner.SparkConnectPlanner.process(SparkConnectPlanner.scala:3901)
	at org.apache.spark.sql.connect.execution.ExecuteThreadRunner.$anonfun$handleCommand$1(ExecuteThreadRunner.scala:567)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.sql.execution.SQLExecution$.withPreExecutionPhaseWatch(SQLExecution.scala:1818)
	at org.apache.spark.sql.connect.execution

The error, `UC_COMMAND_NOT_SUPPORTED_IN_SHARED_ACCESS_MODE`, names the real-time
trigger as the unsupported command. Serverless compute is not supported for
real-time mode
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#prerequisites)).
To run the example for real, use classic compute that meets the requirements
listed above.

## What Real-Time Mode Supports

Because records flow through continuously, not every source, sink and
operation can take part. On Databricks, the current support is as follows
([Real-time mode reference](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)).

| | Supported |
| --- | --- |
| Sources | Apache Kafka, Azure Event Hubs (through the Kafka connector), Amazon Kinesis (EFO mode recommended), Amazon MSK |
| Sinks | Apache Kafka, Azure Event Hubs (through the Kafka connector), custom sinks with `foreach` |
| Stateless operations | Selections, projections, `union` (with some limitations), Scala and Python UDFs (with some limitations) |
| Stateful operations | Aggregations such as `count`, `sum` and `avg`; tumbling and sliding windows; `dropDuplicates` and `dropDuplicatesWithinWatermark`; `transformWithState` |
| Joins | Stream-static joins, broadcast when the table is 10 MB or smaller; stream-stream inner joins on Databricks Runtime 18 and above, with extra configuration |
| Output mode | `update` only |

Notable things that are **not** supported:

- **Delta tables**, as either source or sink. The files, Auto Loader and
  Delta tables used elsewhere in this chapter belong to micro-batch
  streaming. If the data needs to land in a Delta table as well, run a
  second, micro-batch query from the same Kafka topic.
- **`foreachBatch`**, because there are no small batches to hand to it. Use
  `foreach` for custom sinks.
- Session windows, stream-stream outer joins, and stream-static joins
  without a broadcast.
- The `append` and `complete` output modes.

Open-source Apache Spark started with a narrower set: Spark 4.1.0 lists
Scala stateless support
([Apache Spark: Spark Release 4.1.0](https://spark.apache.org/releases/spark-release-4.1.0.html)).
Check the documentation of your Spark version or Databricks Runtime before
relying on a particular operation.

## Sizing a Real-Time Cluster

In micro-batch mode, a cluster with fewer task slots than tasks still works:
the tasks just run in several waves, as described in
{doc}`Distributed Systems Programming Fundamentals <../01_introduction_to_apache_spark/03_distributed_systems_programming_fundamentals/03_distributed_systems_programming_fundamentals>`.
In real-time mode, every stage runs at the same time for the whole batch,
so **every task needs its own slot from the start**. The number of available
task slots must be at least the total number of tasks across all stages of
the query
([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#compute-sizing)),
and compute without enough slots won't schedule the job
([Optimize and monitor real-time mode query performance](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance#compute-tuning)).

:::::{dropdown} Example: How Many Task Slots Does a Query Need?
:open:
:color: primary
:icon: pencil
:class-body: example-body

A real-time query reads from Kafka with `maxPartitions` set to 8. How many
task slots does it need, first as a stateless pipeline and then with an
aggregation added?

**Stateless: Kafka to Kafka, as in the example above.** The query has a
single stage, with one task per source partition:

$$
8 \text{ slots}
$$

**Add a `groupBy` aggregation.** The aggregation needs a shuffle, which adds
a second stage with one task per shuffle partition. With
`spark.sql.shuffle.partitions` set to 20, both stages run concurrently:

$$
8 + 20 = 28 \text{ slots}
$$

On a cluster of workers with 4 cores each, the stateless query fits on 2
workers, while the aggregating query needs $\lceil 28 / 4 \rceil = 7$
workers. Lowering `spark.sql.shuffle.partitions` to match the throughput
you actually need is the easiest way to keep a stateful real-time query
affordable.
:::::

Databricks recommends setting the Kafka source's `maxPartitions` so that
each task handles several Kafka partitions, and tuning
`spark.sql.shuffle.partitions` for shuffle stages, to reach a target
utilisation such as 50%
([Optimize and monitor real-time mode query performance](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance#compute-tuning)).

## Monitoring Latency

In real-time mode, batch durations no longer describe latency, because a
batch lasts minutes by design. Instead, the progress reports carry three
latency metrics, reported as percentiles from P0 to P99
([Optimize and monitor real-time mode query performance](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance#built-in-metrics-with-streamingqueryprogress)):

| Metric | Measures the time from | To |
| --- | --- | --- |
| `sourceQueuingLatencyMs` | A record being written to the message bus | The query reading it |
| `processingLatencyMs` | The query reading a record | Writing it downstream |
| `e2eLatencyMs` | A record being written to the message bus | The query writing it downstream |

High queuing latency points to the source or to a query that cannot keep up.
High processing latency points to an expensive operation inside the query.
The metrics are logged in the driver logs and are available through a
`StreamingQueryListener`'s `onQueryProgress` callback; `busyTimeFraction`
values near 1 mean the tasks are fully utilised and the query may need more
compute
([Optimize and monitor real-time mode query performance](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance#monitoring-and-observability)).

## Delivery Guarantees

Real-time mode keeps Structured Streaming's checkpoints: the batch length
sets how often progress is checkpointed, and a failure replays from the last
checkpoint
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#how-real-time-mode-achieves-low-latency)).
In open-source Spark, the Kafka and `foreach` sinks are **at-least-once**
([Apache Spark: Structured Streaming Programming Guide](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html)),
so records written after the last checkpoint can be written again after a
restart. Consumers of a real-time query's output should therefore tolerate
duplicates, for example by de-duplicating on a key such as `transactionID`,
which the example above uses as the Kafka message key.

:::{admonition} Unverified: Delivery Guarantees of Real-Time Mode
:class: warning

The real-time mode pages on docs.databricks.com don't state its end-to-end
delivery guarantee. The at-least-once behaviour above is taken from the
Apache Spark sink table for micro-batch queries, and couldn't be confirmed
for real-time mode in the official Databricks documentation. Check the
[Real-time mode reference](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)
and its limitations page before relying on it.
:::

## Choosing an Execution Mode

| Latency the business needs | Trigger | Compute |
| --- | --- | --- |
| Hours, or minutes | `availableNow`, scheduled as a job | Runs only while processing; serverless works |
| Seconds to a minute | `processingTime`, or the default trigger | Always running |
| Below a second | `realTime` | Always running; classic compute, sized for all stages at once |

Start from the latency the use case actually needs, not the lowest one
available. Databricks positions real-time mode for sub-second latency and
operational decisions such as blocking fraud, and micro-batch for ETL,
medallion pipelines, cost-sensitive workloads and faster recovery
([Real-time mode in Structured Streaming](https://docs.databricks.com/aws/en/structured-streaming/real-time#when-to-use-real-time-mode)).
Most pipelines that feed tables, dashboards and reports are best served by
scheduled `availableNow` runs, as in the rest of this chapter.

## Common Mistakes

- **Using real-time mode where a scheduled job would do.** It needs
  always-on classic compute with every stage holding its slots, while an
  `availableNow` job runs only when there is data.
- **Leaving Photon, autoscaling or spot instances on.** All three must be
  turned off
  ([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#prerequisites)).
- **Too few task slots.** The cluster needs a slot for every task of every
  stage at once
  ([Set up real-time mode](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup#compute-sizing));
  count them as in the example above.
- **Expecting a Delta source or sink, `foreachBatch`, session windows or
  `append` mode.** None is supported in real-time mode
  ([Real-time mode reference](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)).
- **Reading batch duration as latency.** Use the latency percentiles
  instead
  ([Optimize and monitor real-time mode query performance](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance#built-in-metrics-with-streamingqueryprogress)).

The next chapter,
{doc}`Monitoring and Optimizing Spark Workloads on Databricks <../04_monitoring_and_optimizing_spark_workloads_on_databricks/index>`,
turns from streams to the tools for understanding and tuning any Spark
workload.

## Resources

- [Real-time mode in Structured Streaming (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time)
- [Set up real-time mode (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/setup)
- [Real-time mode reference (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/reference)
- [Optimize and monitor real-time mode query performance (Databricks)](https://docs.databricks.com/aws/en/structured-streaming/real-time/performance)